# 🔭 Observability for a Multi-Agent System

**Assignment:** Transform raw per-agent progress signals into correlated, structured, queryable telemetry.

| | |
|---|---|
| **Runtime** | Python 3.9+ — stdlib only, no installs |
| **Pattern** | Adapted from AWS multipart-upload progress listener |
| **Pipeline** | Planner → Researcher → Writer → Reviewer |

---
### What this notebook covers
1. **Telemetry model** — typed dataclasses for events and summaries  
2. **ObservabilityCollector** — span lifecycle, run/span correlation  
3. **Pluggable sinks** — console (color + progress bar) and JSON Lines  
4. **Orchestrator** — partial-failure semantics  
5. **Live demos** — normal run, failure path, JSON export, event queries


## 1 · Imports

In [ ]:
import random
import time
import json
import uuid
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from typing import Optional, List, Dict, Callable


## 2 · Telemetry model

Three dataclasses capture everything worth knowing about a run:

| Class | Scope | Purpose |
|---|---|---|
| `SpanEvent` | One moment | A single observable event inside one agent |
| `AgentSummary` | One agent | Roll-up after an agent finishes (ok or failed) |
| `RunSummary` | Whole run | Top-level roll-up across all agents |


In [ ]:
@dataclass
class SpanEvent:
    """One observable moment inside a single agent's execution."""
    run_id:        str
    span_id:       str
    agent_name:    str
    event_type:    str           # agent_start | step_complete | agent_end | agent_error
    step:          Optional[int]
    total_steps:   Optional[int]
    progress_pct:  Optional[float]
    elapsed_ms:    Optional[float]
    timestamp:     str
    error:         Optional[str] = None

    def to_dict(self) -> dict:
        return {k: v for k, v in asdict(self).items() if v is not None}


@dataclass
class AgentSummary:
    """Per-agent roll-up computed at the end of a run."""
    agent_name:       str
    span_id:          str
    status:           str        # ok | failed
    steps_completed:  int
    total_steps:      int
    duration_ms:      float
    avg_step_ms:      float
    error:            Optional[str] = None


@dataclass
class RunSummary:
    """Top-level run roll-up."""
    run_id:            str
    status:            str       # ok | partial_failure
    total_agents:      int
    agents_completed:  int
    agents_failed:     int
    total_duration_ms: float
    agent_summaries:   List[AgentSummary]
    started_at:        str
    finished_at:       str

print("✔ Telemetry model defined")


✔ Telemetry model defined


## 3 · Utility helper

In [ ]:
def _iso_now() -> str:
    """UTC timestamp in ISO-8601 with millisecond precision."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

print("✔ Utility defined")


✔ Utility defined


## 4 · ObservabilityCollector

The collector replaces the plain `progress_listener` function.  
It manages **span lifecycle**, **run/span correlation**, and **pluggable sinks**.

```
agent_start  →  step_complete × N  →  agent_end
                                   ↘  agent_error
```


In [ ]:
class ObservabilityCollector:
    """
    Receives raw listener callbacks and builds structured, correlated telemetry.
    One collector per run.
    """

    def __init__(self, run_id: Optional[str] = None):
        self.run_id: str = run_id or str(uuid.uuid4())[:8]
        self._events: List[SpanEvent] = []
        self._span_registry: Dict[str, dict] = {}
        self._run_start: float = time.monotonic()
        self._started_at: str = _iso_now()
        self._sinks: List[Callable[[SpanEvent], None]] = []

    # ── Sink registration ──────────────────────────────────────────────────
    def add_sink(self, sink: Callable[[SpanEvent], None]) -> None:
        self._sinks.append(sink)

    # ── Public listener (drop-in replacement for progress_listener) ────────
    def listener(self, agent_name: str, step: int, total_steps: int) -> None:
        span = self._get_or_create_span(agent_name, total_steps)

        if step == 1:
            self._emit(SpanEvent(
                run_id=self.run_id, span_id=span["span_id"],
                agent_name=agent_name, event_type="agent_start",
                step=None, total_steps=total_steps, progress_pct=0.0,
                elapsed_ms=self._elapsed_ms(), timestamp=_iso_now(),
            ))

        pct = round((step / total_steps) * 100, 1)
        step_elapsed = round((time.monotonic() - span["step_start"]) * 1000, 2)
        span["step_start"] = time.monotonic()
        span["steps_completed"] = step

        self._emit(SpanEvent(
            run_id=self.run_id, span_id=span["span_id"],
            agent_name=agent_name, event_type="step_complete",
            step=step, total_steps=total_steps, progress_pct=pct,
            elapsed_ms=step_elapsed, timestamp=_iso_now(),
        ))

        if step == total_steps:
            span["finished_at"] = time.monotonic()
            self._emit(SpanEvent(
                run_id=self.run_id, span_id=span["span_id"],
                agent_name=agent_name, event_type="agent_end",
                step=total_steps, total_steps=total_steps, progress_pct=100.0,
                elapsed_ms=round((span["finished_at"] - span["started_at"]) * 1000, 2),
                timestamp=_iso_now(),
            ))
            span["status"] = "ok"

    def record_error(self, agent_name: str, error: Exception) -> None:
        span = self._span_registry.get(agent_name)
        if span is None:
            return
        span["status"] = "failed"
        span["error"] = str(error)
        span["finished_at"] = time.monotonic()
        self._emit(SpanEvent(
            run_id=self.run_id, span_id=span["span_id"],
            agent_name=agent_name, event_type="agent_error",
            step=span.get("steps_completed", 0),
            total_steps=span.get("total_steps"),
            progress_pct=None,
            elapsed_ms=round((span["finished_at"] - span["started_at"]) * 1000, 2),
            timestamp=_iso_now(),
            error=str(error),
        ))

    # ── Query helpers ──────────────────────────────────────────────────────
    def events(self,
               agent: Optional[str] = None,
               event_type: Optional[str] = None) -> List[SpanEvent]:
        result = self._events
        if agent:
            result = [e for e in result if e.agent_name == agent]
        if event_type:
            result = [e for e in result if e.event_type == event_type]
        return result

    def build_run_summary(self) -> RunSummary:
        finished_at = _iso_now()
        total_ms = round((time.monotonic() - self._run_start) * 1000, 2)
        agent_summaries = []
        for name, span in self._span_registry.items():
            duration = round(
                ((span.get("finished_at") or time.monotonic()) - span["started_at"]) * 1000, 2
            )
            completed = span.get("steps_completed", 0)
            avg_step = round(duration / completed, 2) if completed else 0.0
            agent_summaries.append(AgentSummary(
                agent_name=name, span_id=span["span_id"],
                status=span.get("status", "unknown"),
                steps_completed=completed, total_steps=span.get("total_steps", 0),
                duration_ms=duration, avg_step_ms=avg_step,
                error=span.get("error"),
            ))
        failed   = sum(1 for s in agent_summaries if s.status == "failed")
        ok_count = sum(1 for s in agent_summaries if s.status == "ok")
        return RunSummary(
            run_id=self.run_id,
            status="ok" if failed == 0 else "partial_failure",
            total_agents=len(self._span_registry),
            agents_completed=ok_count, agents_failed=failed,
            total_duration_ms=total_ms,
            agent_summaries=agent_summaries,
            started_at=self._started_at, finished_at=finished_at,
        )

    def export_events(self) -> List[dict]:
        return [e.to_dict() for e in self._events]

    # ── Internals ──────────────────────────────────────────────────────────
    def _get_or_create_span(self, agent_name: str, total_steps: int) -> dict:
        if agent_name not in self._span_registry:
            self._span_registry[agent_name] = {
                "span_id": str(uuid.uuid4())[:8],
                "started_at": time.monotonic(),
                "step_start": time.monotonic(),
                "total_steps": total_steps,
                "steps_completed": 0,
                "status": "running",
                "error": None,
                "finished_at": None,
            }
        return self._span_registry[agent_name]

    def _emit(self, event: SpanEvent) -> None:
        self._events.append(event)
        for sink in self._sinks:
            sink(event)

    def _elapsed_ms(self) -> float:
        return round((time.monotonic() - self._run_start) * 1000, 2)

print("✔ ObservabilityCollector defined")


✔ ObservabilityCollector defined


## 5 · Sinks

Sinks are callables that receive every `SpanEvent`.  
Add as many as you need — the collector fans out to all of them.

| Sink | Output |
|---|---|
| `ConsoleSink` | Color-coded, progress-bar lines in the notebook cell output |
| `JsonLineSink` | One JSON object per line — ready for Datadog, Loki, CloudWatch |


In [ ]:
class ConsoleSink:
    """Structured, human-readable output with ANSI color and a progress bar."""

    _C = {
        "agent_start":   "\033[36m",   # cyan
        "step_complete": "\033[32m",   # green
        "agent_end":     "\033[34m",   # blue
        "agent_error":   "\033[31m",   # red
        "reset": "\033[0m",
        "dim":   "\033[2m",
        "bold":  "\033[1m",
    }

    def __call__(self, event: SpanEvent) -> None:
        c = self._C
        color = c.get(event.event_type, "")
        R, D, B = c["reset"], c["dim"], c["bold"]

        bar = ""
        if event.event_type in ("step_complete", "agent_end") and event.progress_pct is not None:
            filled = int(event.progress_pct / 10)
            bar = f" [{'█' * filled}{'░' * (10 - filled)}] {event.progress_pct:5.1f}%"

        timing = f"{D}{event.elapsed_ms:7.1f}ms{R}" if event.elapsed_ms else ""

        if event.event_type == "agent_start":
            print(f"{color}▶ START   {B}{event.agent_name:<12}{R}  "
                  f"span={event.span_id}  run={event.run_id}  steps={event.total_steps}")
        elif event.event_type == "step_complete":
            print(f"{color}  step    {event.agent_name:<12}{R}  "
                  f"{event.step:>2}/{event.total_steps}{bar}  {timing}")
        elif event.event_type == "agent_end":
            print(f"{color}✔ END     {B}{event.agent_name:<12}{R}  "
                  f"span={event.span_id}{bar}  total={timing}")
        elif event.event_type == "agent_error":
            print(f"{color}✖ ERROR   {B}{event.agent_name:<12}{R}  "
                  f"span={event.span_id}  at_step={event.step}  {event.error}")


class JsonLineSink:
    """Writes one JSON object per line to a file — pipeline-ready."""

    def __init__(self, path: str):
        self._path = path
        self._fh = open(path, "w", encoding="utf-8")

    def __call__(self, event: SpanEvent) -> None:
        self._fh.write(json.dumps(event.to_dict()) + "\n")
        self._fh.flush()

    def close(self) -> None:
        self._fh.close()

print("✔ ConsoleSink and JsonLineSink defined")


✔ ConsoleSink and JsonLineSink defined


## 6 · Agent & Orchestrator

`Agent` is unchanged from the starter code — pure simulation, no network.  
`Orchestrator` adds **partial-failure semantics**: if one agent errors,  
the remaining agents still run and the failure is captured in the summary.


In [ ]:
class Agent:
    """Simulated agent — does N steps of work with random latency."""

    def __init__(self, name: str, steps: int, fail_at_step: Optional[int] = None):
        self.name = name
        self.steps = steps
        self.fail_at_step = fail_at_step   # set to an int to force a deterministic failure

    def run(self, listener: Callable) -> None:
        for step in range(1, self.steps + 1):
            time.sleep(random.uniform(0.05, 0.2))   # simulate work / latency
            if self.fail_at_step and step == self.fail_at_step:
                raise RuntimeError(f"{self.name} failed at step {step}")
            listener(self.name, step, self.steps)


class Orchestrator:
    def __init__(self, agents: List[Agent], collector: ObservabilityCollector):
        self.agents    = agents
        self.collector = collector

    def run(self) -> RunSummary:
        for agent in self.agents:
            try:
                agent.run(self.collector.listener)
            except Exception as exc:
                # Partial-failure: record error and continue with remaining agents
                self.collector.record_error(agent.name, exc)
        return self.collector.build_run_summary()

print("✔ Agent and Orchestrator defined")


✔ Agent and Orchestrator defined


## 7 · Run report printer

In [ ]:
def print_run_report(summary: RunSummary) -> None:
    SEP = "─" * 62
    ok   = "\033[32m"
    fail = "\033[31m"
    R    = "\033[0m"
    B    = "\033[1m"
    D    = "\033[2m"

    sc = ok if summary.status == "ok" else fail
    print(f"\n{SEP}")
    print(f"{B}Run Report{R}  run_id={summary.run_id}")
    print(SEP)
    print(f"  Status          {sc}{summary.status.upper()}{R}")
    print(f"  Started         {D}{summary.started_at}{R}")
    print(f"  Finished        {D}{summary.finished_at}{R}")
    print(f"  Total duration  {summary.total_duration_ms:.1f} ms")
    print(f"  Agents          {summary.agents_completed}/{summary.total_agents} completed"
          + (f"  {fail}{summary.agents_failed} failed{R}" if summary.agents_failed else ""))
    print()
    print(f"  {'Agent':<14}{'Status':<10}{'Steps':<10}{'Duration':>12}  {'Avg/step':>10}  Error")
    print(f"  {'─'*14}{'─'*10}{'─'*10}{'─'*12}  {'─'*10}  {'─'*20}")
    for s in summary.agent_summaries:
        asc = ok if s.status == "ok" else fail
        print(f"  {s.agent_name:<14}{asc}{s.status:<10}{R}"
              f"{s.steps_completed}/{s.total_steps:<8}"
              f"{s.duration_ms:>10.1f}ms"
              f"  {s.avg_step_ms:>8.1f}ms"
              f"  {s.error or ''}")
    print(SEP)

print("✔ Report printer defined")


✔ Report printer defined


---
## 8 · Demo A — Normal run

All four agents complete successfully.  
Watch the progress bars fill up and the span IDs correlate across events.


In [ ]:
collector_a = ObservabilityCollector()
collector_a.add_sink(ConsoleSink())

agents_normal = [
    Agent("Planner",    3),
    Agent("Researcher", 6),
    Agent("Writer",     4),
    Agent("Reviewer",   2),
]

print(f"\033[1mRun ID: {collector_a.run_id}\033[0m\n")
summary_a = Orchestrator(agents_normal, collector_a).run()
print_run_report(summary_a)


Run ID: 552e70ed

▶ START   Planner       span=ee09009b  run=552e70ed  steps=3
  step    Planner        1/3 [███░░░░░░░]  33.3%      0.1ms
  step    Planner        2/3 [██████░░░░]  66.7%    177.6ms
  step    Planner        3/3 [██████████] 100.0%    103.6ms
✔ END     Planner       span=ee09009b [██████████] 100.0%  total=  281.5ms
▶ START   Researcher    span=6981296d  run=552e70ed  steps=6
  step    Researcher     1/6 [█░░░░░░░░░]  16.7%      0.1ms
  step    Researcher     2/6 [███░░░░░░░]  33.3%    101.3ms
  step    Researcher     3/6 [█████░░░░░]  50.0%     91.3ms
  step    Researcher     4/6 [██████░░░░]  66.7%     69.7ms
  step    Researcher     5/6 [████████░░]  83.3%    110.4ms
  step    Researcher     6/6 [██████████] 100.0%     68.5ms
✔ END     Researcher    span=6981296d [██████████] 100.0%  total=  441.5ms
▶ START   Writer        span=843aa35e  run=552e70ed  steps=4
  step    Writer         1/4 [██░░░░░░░░]  25.0%      0.2ms
  step    Writer         2/4 [█████░░░░░]  50.0% 

## 9 · Demo B — Failure path

Writer is configured to fail at step 2.  
Note that Reviewer still runs — partial-failure semantics in action.


In [ ]:
collector_b = ObservabilityCollector()
collector_b.add_sink(ConsoleSink())

agents_with_failure = [
    Agent("Planner",    3),
    Agent("Researcher", 6),
    Agent("Writer",     4, fail_at_step=2),   # ← will error here
    Agent("Reviewer",   2),
]

print(f"\033[1mRun ID: {collector_b.run_id}\033[0m\n")
summary_b = Orchestrator(agents_with_failure, collector_b).run()
print_run_report(summary_b)


Run ID: edcbb073

▶ START   Planner       span=8deb5037  run=edcbb073  steps=3
  step    Planner        1/3 [███░░░░░░░]  33.3%      0.1ms
  step    Planner        2/3 [██████░░░░]  66.7%    106.4ms
  step    Planner        3/3 [██████████] 100.0%    106.6ms
✔ END     Planner       span=8deb5037 [██████████] 100.0%  total=  213.3ms
▶ START   Researcher    span=40c8cf2f  run=edcbb073  steps=6
  step    Researcher     1/6 [█░░░░░░░░░]  16.7%      0.2ms
  step    Researcher     2/6 [███░░░░░░░]  33.3%    160.8ms
  step    Researcher     3/6 [█████░░░░░]  50.0%     90.2ms
  step    Researcher     4/6 [██████░░░░]  66.7%     70.0ms
  step    Researcher     5/6 [████████░░]  83.3%    193.6ms
  step    Researcher     6/6 [██████████] 100.0%     66.5ms
✔ END     Researcher    span=40c8cf2f [██████████] 100.0%  total=  581.4ms
▶ START   Writer        span=7b0c1d9c  run=edcbb073  steps=4
  step    Writer         1/4 [██░░░░░░░░]  25.0%      0.2ms
✖ ERROR   Writer        span=7b0c1d9c  at_step=1 

## 10 · Demo C — JSON Lines export

Run with both a console sink and a file sink.  
Every event is written as a JSON object on its own line —  
ready to be tailed, shipped to a log aggregator, or queried with `jq`.


In [ ]:
EXPORT_PATH = "events.jsonl"

collector_c  = ObservabilityCollector()
json_sink    = JsonLineSink(EXPORT_PATH)

collector_c.add_sink(ConsoleSink())
collector_c.add_sink(json_sink)

agents_export = [
    Agent("Planner",    3),
    Agent("Researcher", 6),
    Agent("Writer",     4),
    Agent("Reviewer",   2),
]

print(f"\033[1mRun ID: {collector_c.run_id}\033[0m\n")
summary_c = Orchestrator(agents_export, collector_c).run()
json_sink.close()
print(f"\n→ Events written to: {EXPORT_PATH}")


Run ID: 97fa2413

▶ START   Planner       span=3f661182  run=97fa2413  steps=3
  step    Planner        1/3 [███░░░░░░░]  33.3%      0.2ms
  step    Planner        2/3 [██████░░░░]  66.7%    168.7ms
  step    Planner        3/3 [██████████] 100.0%    177.9ms
✔ END     Planner       span=3f661182 [██████████] 100.0%  total=  347.0ms
▶ START   Researcher    span=f1bba650  run=97fa2413  steps=6
  step    Researcher     1/6 [█░░░░░░░░░]  16.7%      0.5ms
  step    Researcher     2/6 [███░░░░░░░]  33.3%    156.4ms
  step    Researcher     3/6 [█████░░░░░]  50.0%    140.9ms
  step    Researcher     4/6 [██████░░░░]  66.7%    182.2ms
  step    Researcher     5/6 [████████░░]  83.3%     76.4ms
  step    Researcher     6/6 [██████████] 100.0%    120.9ms
✔ END     Researcher    span=f1bba650 [██████████] 100.0%  total=  677.7ms
▶ START   Writer        span=f2448902  run=97fa2413  steps=4
  step    Writer         1/4 [██░░░░░░░░]  25.0%      0.5ms
  step    Writer         2/4 [█████░░░░░]  50.0% 

## 11 · Inspect the exported JSON Lines

Read back the file and pretty-print the first few events.


In [ ]:
with open(EXPORT_PATH) as f:
    raw_events = [json.loads(line) for line in f]

print(f"Total events exported: {len(raw_events)}\n")
print("─── First 3 events ───")
for ev in raw_events[:3]:
    print(json.dumps(ev, indent=2))


Total events exported: 23

─── First 3 events ───
{
  "run_id": "97fa2413",
  "span_id": "3f661182",
  "agent_name": "Planner",
  "event_type": "agent_start",
  "total_steps": 3,
  "progress_pct": 0.0,
  "elapsed_ms": 134.5,
  "timestamp": "2026-06-24T05:55:52.912Z"
}
{
  "run_id": "97fa2413",
  "span_id": "3f661182",
  "agent_name": "Planner",
  "event_type": "step_complete",
  "step": 1,
  "total_steps": 3,
  "progress_pct": 33.3,
  "elapsed_ms": 0.24,
  "timestamp": "2026-06-24T05:55:52.912Z"
}
{
  "run_id": "97fa2413",
  "span_id": "3f661182",
  "agent_name": "Planner",
  "event_type": "step_complete",
  "step": 2,
  "total_steps": 3,
  "progress_pct": 66.7,
  "elapsed_ms": 168.67,
  "timestamp": "2026-06-24T05:55:53.081Z"
}


## 12 · Query the collector

The collector exposes a simple query API —  
filter by agent name, event type, or both.


In [ ]:
# All error events from the failure run
errors = collector_b.events(event_type="agent_error")
print(f"Error events in run {collector_b.run_id}: {len(errors)}")
for e in errors:
    print(f"  agent={e.agent_name}  step={e.step}  error={e.error}")

print()

# All steps for Researcher in the normal run
researcher_steps = collector_a.events(agent="Researcher", event_type="step_complete")
print(f"Researcher steps in run {collector_a.run_id}:")
for e in researcher_steps:
    print(f"  step {e.step}/{e.total_steps}  {e.progress_pct}%  {e.elapsed_ms}ms")


Error events in run edcbb073: 1
  agent=Writer  step=1  error=Writer failed at step 2

Researcher steps in run 552e70ed:
  step 1/6  16.7%  0.14ms
  step 2/6  33.3%  101.26ms
  step 3/6  50.0%  91.32ms
  step 4/6  66.7%  69.69ms
  step 5/6  83.3%  110.41ms
  step 6/6  100.0%  68.54ms


## 13 · RunSummary as JSON

Serialize the run summary — useful for storing in a database,  
returning from an API endpoint, or feeding into a dashboard.


In [ ]:
import dataclasses

def summary_to_dict(summary: RunSummary) -> dict:
    d = dataclasses.asdict(summary)
    return d

summary_dict = summary_to_dict(summary_a)
print(json.dumps(summary_dict, indent=2))


{
  "run_id": "552e70ed",
  "status": "ok",
  "total_agents": 4,
  "agents_completed": 4,
  "agents_failed": 0,
  "total_duration_ms": 1664.78,
  "agent_summaries": [
    {
      "agent_name": "Planner",
      "span_id": "ee09009b",
      "status": "ok",
      "steps_completed": 3,
      "total_steps": 3,
      "duration_ms": 281.46,
      "avg_step_ms": 93.82,
      "error": null
    },
    {
      "agent_name": "Researcher",
      "span_id": "6981296d",
      "status": "ok",
      "steps_completed": 6,
      "total_steps": 6,
      "duration_ms": 441.5,
      "avg_step_ms": 73.58,
      "error": null
    },
    {
      "agent_name": "Writer",
      "span_id": "843aa35e",
      "status": "ok",
      "steps_completed": 4,
      "total_steps": 4,
      "duration_ms": 446.9,
      "avg_step_ms": 111.72,
      "error": null
    },
    {
      "agent_name": "Reviewer",
      "span_id": "d29a29fd",
      "status": "ok",
      "steps_completed": 2,
      "total_steps": 2,
      "duration_ms"

---
## 14 · Key takeaways

| Concept | Implementation |
|---|---|
| **Span correlation** | Every event carries `run_id` + `span_id` — one query to find all events for one agent in one run |
| **Structured events** | Typed dataclasses, not strings — filterable, serialisable, schema-stable |
| **Pluggable sinks** | Observer pattern — add console, file, HTTP, Kafka sinks without touching core logic |
| **Partial failure** | Orchestrator catches per-agent errors, records them, and continues — no silent crashes |
| **Queryable telemetry** | `collector.events(agent=..., event_type=...)` — same idea as a log query language, but in-process |

This pattern transplants directly onto a real LLM agent pipeline:  
replace `time.sleep(...)` with your actual agent call and the rest stays the same.
